In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

order_level = pd.read_csv("../data/processed/order_level_dataset.csv")

order_level.head()
order_level.shape

date_columns = [
    "purchase_timestamp",
    "approved_at",
    "delivered_carrier_date",
    "delivered_customer_date",
    "estimated_delivery_date"
]

for col in date_columns:
    order_level[col] = pd.to_datetime(order_level[col], errors="coerce")

order_level[date_columns].dtypes

In [ ]:
#Basic business KPIs
total_orders = order_level["order_id"].nunique()
total_customers = order_level["customer_unique_id"].nunique()
total_revenue = order_level["total_payment"].sum()
avg_order_value = order_level["total_payment"].mean()
avg_review_score = order_level["review_score"].mean()
late_delivery_rate = order_level["is_late"].mean() * 100
avg_delivery_days = order_level["delivery_days"].mean()

print("Total Orders:", total_orders)
print("Total Customers:", total_customers)
print("Total Revenue:", round(total_revenue, 2))
print("Average Order Value:", round(avg_order_value, 2))
print("Average Review Score:", round(avg_review_score, 2))
print("Late Delivery Rate (%):", round(late_delivery_rate, 2))
print("Average Delivery Days:", round(avg_delivery_days, 2))

#KPIs summary
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Orders",
        "Total Customers",
        "Total Revenue",
        "Average Order Value",
        "Average Review Score",
        "Late Delivery Rate (%)",
        "Average Delivery Days"
    ],
    "Value": [
        total_orders,
        total_customers,
        round(total_revenue, 2),
        round(avg_order_value, 2),
        round(avg_review_score, 2),
        round(late_delivery_rate, 2),
        round(avg_delivery_days, 2)
    ]
})

kpi_summary

kpi_summary.to_csv("../reports/phase_3_kpi_summary.csv", index=False)

In [ ]:
#Revenue trend by month
monthly_revenue = (
    order_level
    .groupby("purchase_year_month")
    .agg(
        total_revenue=("total_payment", "sum"),
        total_orders=("order_id", "nunique"),
        avg_order_value=("total_payment", "mean")
    )
    .reset_index()
)

monthly_revenue.head()

#Plotting monthly revenue trend
plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_revenue, x="purchase_year_month", y="total_revenue", marker="o")
plt.xticks(rotation=45)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

monthly_revenue.to_csv("../reports/monthly_revenue.csv", index=False)

In [ ]:
#Orders trend by month
#Are sales increasing or decreasing over time? This can indicate seasonality, growth, or decline in business performance.
plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_revenue, x="purchase_year_month", y="total_orders", marker="o")
plt.xticks(rotation=45)
plt.title("Monthly Order Trend")
plt.xlabel("Month")
plt.ylabel("Total Orders")
plt.tight_layout()
plt.show()

In [ ]:
#Revenue by customer state
state_revenue = (
    order_level
    .groupby("customer_state")
    .agg(
        total_revenue=("total_payment", "sum"),
        total_orders=("order_id", "nunique"),
        avg_order_value=("total_payment", "mean"),
        avg_review_score=("review_score", "mean"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

state_revenue["late_delivery_rate"] = state_revenue["late_delivery_rate"] * 100

state_revenue = state_revenue.sort_values("total_revenue", ascending=False)

state_revenue.head(10)

#Plotting revenue by customer state
top_states = state_revenue.head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_states, x="customer_state", y="total_revenue")
plt.title("Top 10 States by Revenue")
plt.xlabel("Customer State")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

state_revenue.to_csv("../reports/state_revenue_analysis.csv", index=False)

In [ ]:
#Product category revenue
category_revenue = (
    order_level
    .groupby("main_product_category")
    .agg(
        total_revenue=("total_payment", "sum"),
        total_orders=("order_id", "nunique"),
        avg_order_value=("total_payment", "mean"),
        avg_review_score=("review_score", "mean"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

category_revenue["late_delivery_rate"] = category_revenue["late_delivery_rate"] * 100

category_revenue = category_revenue.sort_values("total_revenue", ascending=False)

category_revenue.head(15)

#Plotting revenue by product category
top_categories = category_revenue.head(15)

plt.figure(figsize=(14, 7))
sns.barplot(data=top_categories, y="main_product_category", x="total_revenue")
plt.title("Top 15 Product Categories by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Product Category")
plt.tight_layout()
plt.show()

category_revenue.to_csv("../reports/category_revenue_analysis.csv", index=False)

In [ ]:
#Review score distribution
review_distribution = (
    order_level["review_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)

review_distribution.columns = ["review_score", "order_count"]

review_distribution

plt.figure(figsize=(8, 5))
sns.barplot(data=review_distribution, x="review_score", y="order_count")
plt.title("Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

review_distribution.to_csv("../reports/review_score_distribution.csv", index=False)

In [ ]:
#Late delivery analysis
late_delivery_summary = (
    order_level
    .groupby("is_late")
    .agg(
        total_orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        avg_delay_days=("delay_days", "mean"),
        avg_delivery_days=("delivery_days", "mean")
    )
    .reset_index()
)

late_delivery_summary

late_delivery_summary["delivery_status"] = late_delivery_summary["is_late"].map({
    0: "On Time",
    1: "Late"
})

late_delivery_summary

plt.figure(figsize=(7, 5))
sns.barplot(data=late_delivery_summary, x="delivery_status", y="avg_review_score")
plt.title("Average Review Score: On-Time vs Late Deliveries")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.tight_layout()
plt.show()

late_delivery_summary.to_csv("../reports/late_delivery_summary.csv", index=False)

In [ ]:
#Late delivery rate by state
late_by_state = (
    order_level
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "nunique"),
        late_delivery_rate=("is_late", "mean"),
        avg_delay_days=("delay_days", "mean"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

late_by_state["late_delivery_rate"] = late_by_state["late_delivery_rate"] * 100

late_by_state = late_by_state.sort_values("late_delivery_rate", ascending=False)

late_by_state.head(10)

top_late_states = late_by_state[late_by_state["total_orders"] >= 100].head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_late_states, x="customer_state", y="late_delivery_rate")
plt.title("Top States by Late Delivery Rate")
plt.xlabel("Customer State")
plt.ylabel("Late Delivery Rate (%)")
plt.tight_layout()
plt.show()

late_by_state.to_csv("../reports/late_delivery_by_state.csv", index=False)

In [ ]:
#Payment type analysis
payment_analysis = (
    order_level
    .groupby("payment_type")
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("total_payment", "sum"),
        avg_order_value=("total_payment", "mean"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

payment_analysis

plt.figure(figsize=(10, 5))
sns.barplot(data=payment_analysis, x="payment_type", y="total_revenue")
plt.title("Revenue by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

payment_analysis.to_csv("../reports/payment_type_analysis.csv", index=False)

In [ ]:
#Customer repeat purchase analysis
customer_order_counts = (
    order_level
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spent=("total_payment", "sum"),
        avg_order_value=("total_payment", "mean")
    )
    .reset_index()
)

customer_order_counts.head()

#Repaet vs one-time
customer_order_counts["customer_type"] = np.where(
    customer_order_counts["total_orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

customer_type_summary = (
    customer_order_counts
    .groupby("customer_type")
    .agg(
        customer_count=("customer_unique_id", "nunique"),
        total_orders=("total_orders", "sum"),
        total_revenue=("total_spent", "sum"),
        avg_order_value=("avg_order_value", "mean")
    )
    .reset_index()
)

customer_type_summary

plt.figure(figsize=(7, 5))
sns.barplot(data=customer_type_summary, x="customer_type", y="customer_count")
plt.title("One-Time vs Repeat Customers")
plt.xlabel("Customer Type")
plt.ylabel("Customer Count")
plt.tight_layout()
plt.show()

customer_type_summary.to_csv("../reports/customer_type_summary.csv", index=False)

In [ ]:
#Top customers by revenue
top_customers = customer_order_counts.sort_values(
    "total_spent",
    ascending=False
).head(20)

top_customers

top_customers.to_csv("../reports/top_customers_by_revenue.csv", index=False)

In [ ]:
#Delivery day distribution
plt.figure(figsize=(10, 5))
sns.histplot(order_level["delivery_days"], bins=50, kde=True)
plt.title("Delivery Days Distribution")
plt.xlabel("Delivery Days")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

order_level["delivery_days"].describe()



In [ ]:
#Revenue vs review score
review_revenue = (
    order_level
    .groupby("review_score")
    .agg(
        total_orders=("order_id", "nunique"),
        avg_order_value=("total_payment", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

review_revenue["late_delivery_rate"] = review_revenue["late_delivery_rate"] * 100

review_revenue

plt.figure(figsize=(8, 5))
sns.barplot(data=review_revenue, x="review_score", y="late_delivery_rate")
plt.title("Late Delivery Rate by Review Score")
plt.xlabel("Review Score")
plt.ylabel("Late Delivery Rate (%)")
plt.tight_layout()
plt.show()

review_revenue.to_csv("../reports/review_revenue_analysis.csv", index=False)

In [22]:
#Business insight summary
business_insights = [
    {
        "Insight Area": "Revenue",
        "Insight": "Revenue is concentrated among the top customer states and product categories.",
        "Business Recommendation": "Prioritize marketing and logistics investment in high-revenue states and categories."
    },
    {
        "Insight Area": "Delivery",
        "Insight": "Late deliveries have lower average review scores than on-time deliveries.",
        "Business Recommendation": "Improve delivery reliability in states with high late delivery rates."
    },
    {
        "Insight Area": "Customer",
        "Insight": "Most customers are one-time buyers.",
        "Business Recommendation": "Create retention campaigns for first-time customers."
    },
    {
        "Insight Area": "Product",
        "Insight": "Some high-revenue categories may still have lower review scores or higher delay rates.",
        "Business Recommendation": "Monitor category-level satisfaction and logistics performance."
    },
    {
        "Insight Area": "Payment",
        "Insight": "Payment method behavior differs by revenue contribution and order value.",
        "Business Recommendation": "Optimize checkout and installment options for high-value payment types."
    }
]

business_insights_df = pd.DataFrame(business_insights)
business_insights_df

business_insights_df.to_csv("../reports/phase_3_business_insights.csv", index=False)


In [23]:
order_level.to_csv("../data/processed/order_level_eda_ready.csv", index=False)

# Phase 3 Conclusion

In Phase 3, exploratory data analysis was performed on the cleaned order-level dataset.

Main work completed:

- Created core business KPIs.
- Analyzed monthly revenue and order trends.
- Identified top states by revenue.
- Identified top product categories by revenue.
- Analyzed review score distribution.
- Compared customer satisfaction between late and on-time deliveries.
- Analyzed late delivery rate by customer state.
- Analyzed payment type performance.
- Identified one-time and repeat customer behavior.
- Created business recommendation summaries.

Key business themes:

- Revenue is concentrated in specific states and product categories.
- Delivery performance directly affects customer satisfaction.
- Most customers are one-time buyers, creating a retention opportunity.
- Customer geography, product category, payment behavior, and delivery speed are important business drivers.

The outputs from this phase will support Power BI dashboarding, SQL analytics, and machine learning feature selection.